# Comparativa: IA Local vs Cloud

> Mide y compara latencia, coste y calidad entre modelos ejecutados localmente (Ollama)
> y modelos en la nube (Claude API). Aprende a diseñar un router híbrido inteligente.

## Objetivos
- Medir latencia real de llamadas locales vs API
- Estimar costes por petición en cada opción
- Construir un router que elije automáticamente según el tipo de tarea
- Visualizar la comparativa con pandas y matplotlib

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic requests pandas matplotlib --quiet

## 2. Configuración y preguntas de prueba

In [ ]:
import anthropic
import requests
import time
import pandas as pd
import matplotlib.pyplot as plt

cliente_anthropic = anthropic.Anthropic()
MODELO_CLOUD = "claude-haiku-4-5-20251001"
OLLAMA_URL = "http://localhost:11434"
MODELO_LOCAL = "llama3.2"  # Cambia por el modelo que tengas en Ollama

# Verificar disponibilidad de Ollama
ollama_disponible = False
try:
    r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
    if r.status_code == 200:
        modelos = [m["name"] for m in r.json().get("models", [])]
        if modelos:
            MODELO_LOCAL = modelos[0]
            ollama_disponible = True
            print(f"✓ Ollama disponible. Modelo: {MODELO_LOCAL}")
except Exception:
    print("✗ Ollama no disponible. Solo se ejecutarán pruebas con la API de Anthropic.")

print(f"✓ API Anthropic configurada. Modelo: {MODELO_CLOUD}")

PREGUNTAS = [
    "¿Qué es Python?",
    "Resume en 3 puntos qué es el machine learning.",
    "¿Cuáles son las ventajas de usar PostgreSQL frente a MySQL?"
]

## 3. Medir latencia: Cloud (Anthropic API)

In [ ]:
def llamar_cloud(pregunta: str) -> dict:
    """Llama a Claude Haiku y mide latencia y tokens."""
    inicio = time.perf_counter()
    respuesta = cliente_anthropic.messages.create(
        model=MODELO_CLOUD,
        max_tokens=256,
        messages=[{"role": "user", "content": pregunta}]
    )
    latencia = time.perf_counter() - inicio
    tokens_entrada = respuesta.usage.input_tokens
    tokens_salida = respuesta.usage.output_tokens
    # Precio Haiku: $0.25/MTok entrada, $1.25/MTok salida (abril 2025)
    coste_usd = (tokens_entrada * 0.00000025) + (tokens_salida * 0.00000125)
    return {
        "proveedor": "Anthropic Claude Haiku",
        "latencia_s": round(latencia, 3),
        "tokens_salida": tokens_salida,
        "coste_usd": round(coste_usd, 6),
        "respuesta": respuesta.content[0].text[:200]
    }

print("=== PRUEBA CON CLOUD (Anthropic Claude Haiku) ===")
resultados_cloud = []
for p in PREGUNTAS:
    r = llamar_cloud(p)
    resultados_cloud.append(r)
    print(f"  Pregunta: '{p[:50]}'")
    print(f"  Latencia: {r['latencia_s']}s | Tokens: {r['tokens_salida']} | Coste: ${r['coste_usd']:.5f}")

## 4. Medir latencia: Local (Ollama)

In [ ]:
def llamar_local(pregunta: str) -> dict:
    """Llama a Ollama local y mide latencia."""
    if not ollama_disponible:
        # Simular con latencias típicas si Ollama no está disponible
        import random
        latencia = random.uniform(2.5, 6.0)  # Latencias típicas en CPU
        return {
            "proveedor": f"Ollama {MODELO_LOCAL} [simulado]",
            "latencia_s": round(latencia, 3),
            "tokens_salida": random.randint(80, 200),
            "coste_usd": 0.0,
            "respuesta": "[Ollama no disponible — dato simulado]"
        }

    inicio = time.perf_counter()
    payload = {"model": MODELO_LOCAL, "messages": [{"role": "user", "content": pregunta}], "stream": False}
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=120)
    latencia = time.perf_counter() - inicio
    datos = r.json()
    tokens_salida = datos.get("eval_count", 0)
    return {
        "proveedor": f"Ollama {MODELO_LOCAL}",
        "latencia_s": round(latencia, 3),
        "tokens_salida": tokens_salida,
        "coste_usd": 0.0,  # Local = sin coste por token
        "respuesta": datos.get("message", {}).get("content", "")[:200]
    }

print(f"=== PRUEBA CON LOCAL (Ollama {MODELO_LOCAL}) ===")
resultados_local = []
for p in PREGUNTAS:
    r = llamar_local(p)
    resultados_local.append(r)
    print(f"  Pregunta: '{p[:50]}'")
    print(f"  Latencia: {r['latencia_s']}s | Coste: ${r['coste_usd']}")

## 5. Tabla comparativa y visualización

In [ ]:
# Tabla comparativa con pandas
resumen = pd.DataFrame([
    {"Proveedor": "Anthropic Claude Haiku", "Latencia media (s)": round(sum(r['latencia_s'] for r in resultados_cloud)/len(resultados_cloud), 3),
     "Coste total (USD)": round(sum(r['coste_usd'] for r in resultados_cloud), 6), "Privacidad": "❌ Datos al exterior", "Calidad": "★★★★★"},
    {"Proveedor": f"Ollama {MODELO_LOCAL}", "Latencia media (s)": round(sum(r['latencia_s'] for r in resultados_local)/len(resultados_local), 3),
     "Coste total (USD)": 0.0, "Privacidad": "✅ Datos en local", "Calidad": "★★★☆☆"},
])
print("=== COMPARATIVA ===")
print(resumen.to_string(index=False))

# Gráfica de latencias
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

latencias_cloud = [r['latencia_s'] for r in resultados_cloud]
latencias_local = [r['latencia_s'] for r in resultados_local]
x = range(len(PREGUNTAS))
etiquetas = [f"P{i+1}" for i in x]

axes[0].bar([xi - 0.2 for xi in x], latencias_cloud, width=0.4, label="Anthropic Haiku", color="#3498db", alpha=0.8)
axes[0].bar([xi + 0.2 for xi in x], latencias_local, width=0.4, label=f"Ollama {MODELO_LOCAL}", color="#2ecc71", alpha=0.8)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(etiquetas)
axes[0].set_title("Latencia por Pregunta (segundos)", fontweight="bold")
axes[0].set_ylabel("Segundos")
axes[0].legend()

# Tabla de dimensiones
dimensiones = ["Latencia", "Coste", "Privacidad", "Calidad", "Offline"]
cloud_scores = [8, 5, 3, 9, 1]
local_scores = [5, 10, 10, 6, 10]
axes[1].barh(dimensiones, cloud_scores, 0.4, label="Cloud", color="#3498db", alpha=0.8)
axes[1].barh([d + "  " for d in dimensiones], local_scores, 0.4, label="Local", color="#2ecc71", alpha=0.8)
axes[1].set_xlim(0, 12)
axes[1].set_title("Puntuación por Dimensión (1-10)", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("comparativa_local_cloud.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Router híbrido inteligente

In [ ]:
def router_inteligente(pregunta: str, contiene_pii: bool = False, requiere_maxima_calidad: bool = False) -> dict:
    """
    Elige automáticamente entre local y cloud según el tipo de tarea:
    - Datos sensibles (PII) → siempre local
    - Máxima calidad requerida → siempre cloud
    - Preguntas simples/offline → local si disponible
    - Default → cloud por calidad
    """
    if contiene_pii:
        decision = "local"
        motivo = "La pregunta contiene datos sensibles — se procesa en local"
    elif requiere_maxima_calidad:
        decision = "cloud"
        motivo = "Se requiere máxima calidad — usando API cloud"
    elif ollama_disponible and len(pregunta) < 200:
        decision = "local"
        motivo = "Pregunta simple y Ollama disponible — procesando en local"
    else:
        decision = "cloud"
        motivo = "Default: usando API cloud"

    print(f"Decisión: {decision.upper()} — {motivo}")
    if decision == "local":
        return llamar_local(pregunta)
    else:
        return llamar_cloud(pregunta)

# Casos de uso del router
casos = [
    ("¿Qué es el transfer learning?", False, False),
    ("El paciente Juan García, DNI 12345678A, tiene historial de...", True, False),
    ("Analiza este contrato legal complejo y detecta cláusulas abusivas.", False, True),
]

print("=== ROUTER HÍBRIDO EN ACCIÓN ===")
for pregunta, pii, calidad in casos:
    print(f"\nPregunta: '{pregunta[:60]}...'")
    resultado = router_inteligente(pregunta, pii, calidad)
    print(f"Proveedor usado: {resultado['proveedor']} | Latencia: {resultado['latencia_s']}s")